# LLM 01: Neural Net Refresher

Build a tiny neural net from scratch (NumPy), then the same thing in PyTorch, then preview the transformer's core idea: self-attention.

**Run order:** top to bottom. Dependencies: `numpy`, `torch`. Install with `pip install numpy torch`.

## 1. NumPy: a 2-layer NN learns XOR

XOR is the classic non-linearly-separable problem. A single linear layer cannot solve it; two layers with a non-linearity can.

In [ ]:
import numpy as np
np.random.seed(0)

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y = np.array([[0], [1], [1], [0]], dtype=float)

hidden = 4
W1 = np.random.randn(2, hidden) * 0.5
b1 = np.zeros((1, hidden))
W2 = np.random.randn(hidden, 1) * 0.5
b2 = np.zeros((1, 1))

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

lr = 0.5
for epoch in range(10_000):
    h = np.tanh(X @ W1 + b1)
    pred = sigmoid(h @ W2 + b2)

    loss = -np.mean(y * np.log(pred + 1e-9) + (1 - y) * np.log(1 - pred + 1e-9))

    dpred = (pred - y) / len(X)
    dW2 = h.T @ dpred
    db2 = dpred.sum(axis=0, keepdims=True)
    dh = dpred @ W2.T * (1 - h ** 2)
    dW1 = X.T @ dh
    db1 = dh.sum(axis=0, keepdims=True)

    W1 -= lr * dW1; b1 -= lr * db1
    W2 -= lr * dW2; b2 -= lr * db2

    if epoch % 2000 == 0:
        print(f"epoch {epoch:5d}  loss={loss:.4f}")

print("\nFinal predictions (rounded):")
print(np.round(pred, 2))

## 2. PyTorch: same model, automated gradients

PyTorch handles backprop for us via autograd. Notice how much less code it takes.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)
X = torch.tensor([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=torch.float32)
y = torch.tensor([[0], [1], [1], [0]], dtype=torch.float32)

model = nn.Sequential(
    nn.Linear(2, 4),
    nn.Tanh(),
    nn.Linear(4, 1),
    nn.Sigmoid(),
)
opt = torch.optim.SGD(model.parameters(), lr=0.5)
loss_fn = nn.BCELoss()

for epoch in range(10_000):
    pred = model(X)
    loss = loss_fn(pred, y)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if epoch % 2000 == 0:
        print(f"epoch {epoch:5d}  loss={loss.item():.4f}")

with torch.no_grad():
    print("\nFinal predictions:", model(X).squeeze().round(decimals=2).tolist())

## 3. From FC layers to attention

A fully-connected layer mixes all inputs with a fixed weight matrix. **Self-attention** is a *content-dependent* mix: how much token `i` attends to token `j` depends on their embeddings.

The simplified formula:

```
attn(Q, K, V) = softmax(Q @ K.T / sqrt(d)) @ V
```

Every token produces a Query, Key, and Value vector. The Query asks *who should I pay attention to?* — softmax over dot-products with every Key. Those weights mix the Values into a new representation.

Below: a minimal self-attention block, just enough to see the shapes flow.

In [ ]:
import torch
import torch.nn.functional as F

torch.manual_seed(0)
seq_len, d_model = 4, 8   # 4 tokens, 8-dim embeddings
x = torch.randn(seq_len, d_model)

Wq = torch.randn(d_model, d_model)
Wk = torch.randn(d_model, d_model)
Wv = torch.randn(d_model, d_model)

Q = x @ Wq
K = x @ Wk
V = x @ Wv

scores = Q @ K.T / (d_model ** 0.5)
weights = F.softmax(scores, dim=-1)
out = weights @ V

print("attention weights (rows sum to 1):")
print(weights.round(decimals=2))
print("\noutput shape:", out.shape)

## 4. Mental model takeaways

| Concept | One-line intuition |
| --- | --- |
| Depth | Compositional features |
| Attention | Content-dependent mixing of tokens |
| Scaling laws | Bigger model + more data = lower loss, predictably |
| Pretrain → SFT → RLHF | Capability → Instructability → Preference alignment |

Next up: **02 — Transformer Architecture**, where we dissect multi-head attention, positional encoding (RoPE), and how GPT/Claude/Llama actually stack these blocks.